# DS2001 Final Project
Aaditi Parab

### Importing required libraries

In [143]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

C:\Program Files\spark-3.5.4-bin-hadoop3


### Instantiating connection variables

In [144]:
#information for connecting to mysql server
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "adventureworks",
    "conn_props" : {
        "user" : "root",
        "password" : "#Hi10172004",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

#mongodb connection information
mongodb_args = {
    "cluster_location" : "local",
    "user_name" : "",
    "password" : "",
    "cluster_name" : "",
    "cluster_subnet" : "",
    "db_name" : "adventureworks",
    "collection" : "",
    "null_column_threshold" : 0.5
}

#info about where local batch data is stored
base_dir = os.path.join(os.getcwd(), 'proj_data')

batch_dir = os.path.join(base_dir, 'batch')
stream_dir = os.path.join(base_dir, 'streaming')

sales_orders_stream_dir = os.path.join(stream_dir, 'sales_orders')

#creating initial data lakehouse
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

sales_orders_output_bronze = os.path.join(database_dir, 'fact_sales_orders', 'bronze')
sales_orders_output_silver = os.path.join(database_dir, 'fact_sales_orders', 'silver')
sales_orders_output_gold   = os.path.join(database_dir, 'fact_sales_orders', 'gold')

### defining functions that will be used later

In [145]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))
    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name', 'size', 'modification_time']
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
    print(f"The stream has processed {len(query.recentProgress)} batch(es)")


def remove_directory_tree(path: str):
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
    except Exception as e:
        return f"An error occurred: {e}"


def drop_null_columns(df, threshold):
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold]
    df_dropped = df.drop(*columns_with_nulls)
    return df_dropped


def get_mysql_dataframe(spark_session, sql_query: str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    dframe = spark_session.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("driver", args['conn_props']['driver']) \
        .option("user", args['conn_props']['user']) \
        .option("password", args['conn_props']['password']) \
        .option("query", sql_query) \
        .load()
    return dframe


def get_mongo_uri(**args):
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"
    return uri


def get_mongo_client(**args):
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())
    else:
        client = pymongo.MongoClient(mongo_uri)
    return client


def set_mongo_collections(mongo_client, db_name: str, data_directory: str, json_files: dict):
    db = mongo_client[db_name]
    for collection_name, filename in json_files.items():
        db.drop_collection(collection_name)
        json_file = os.path.join(data_directory, filename)
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
        collection = db[collection_name]
        collection.insert_many(json_object)
    mongo_client.close()


def get_mongodb_dataframe(spark_session, **args):
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()
    dframe = dframe.drop('_id')
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    return dframe

### removing database for idempotency

In [146]:
print(remove_directory_tree(database_dir))

Directory 'C:\Users\aadit\Downloads\dssystems\DS-2002\FinalProj\spark-warehouse\adventureworks_dlh.db' has been removed successfully.


### start new spark session

In [147]:
mysql_spark_jar = os.path.join(os.getcwd(), 'mysql-connector-j-9.1.0', 'mysql-connector-j-9.1.0.jar')

mongo_uri = get_mongo_uri(**mongodb_args)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DS-2002 Final Project – AdventureWorks Data Lakehouse")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.jars", mysql_spark_jar)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1")
    .config("spark.mongodb.input.uri", mongo_uri)
    .config("spark.mongodb.output.uri", mongo_uri)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", str(int(os.cpu_count())))
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
    .config("spark.sql.streaming.schemaInference", "true")
    .config("spark.sql.warehouse.dir", sql_warehouse_dir)
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("OFF")
spark

### creating dest database

In [148]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Final Project – AdventureWorks Data Lakehouse'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Final Project');
"""
spark.sql(sql_create_db)

DataFrame[]

In [149]:
get_file_info(batch_dir)

,name,size,modification_time
0,dim_customers.json,478186,2026-05-08 09:35:06.224101543
1,dim_employee.csv,53785,2026-05-08 09:36:19.275834084


### reading in data from employee_csv file

In [150]:
employee_csv = os.path.join(batch_dir, 'dim_employee.csv')
print(employee_csv)

df_dim_employees = spark.read.format('csv').options(header='true', inferSchema='true').load(employee_csv)
#df_dim_employees.toPandas().head(2)

C:\Users\aadit\Downloads\dssystems\DS-2002\FinalProj\proj_data\batch\dim_employee.csv


### make transformations

In [151]:
# rename id
df_dim_employees = df_dim_employees.withColumnRenamed("EmployeeID", "employee_id")

# surrogate primary key, ROW_NUMBER()
df_dim_employees.createOrReplaceTempView("employees")
sql_employees = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY employee_id) AS employee_key
    FROM employees;
"""
df_dim_employees = spark.sql(sql_employees)

ordered_columns = ['employee_key', 'employee_id', 'first_name', 'last_name',
                   'job_title', 'department', 'hire_date', 'email']

# keep matching columns
existing_columns = [c for c in ordered_columns if c in df_dim_employees.columns]
df_dim_employees = df_dim_employees[existing_columns]
df_dim_employees.toPandas().head(2)

,employee_key,employee_id
0,1,1
1,2,2


In [152]:
df_dim_employees

DataFrame[employee_key: int, employee_id: int]

### save into lakehouse as dim employees

In [153]:
df_dim_employees.write.saveAsTable(f"{dest_database}.dim_employees", mode="overwrite")

### view table to check

In [154]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_employees;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_employees LIMIT 5").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        employee_key|                 int|   NULL|
|         employee_id|                 int|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|       dim_employees|       |
|        Created Time|Fri May 08 23:43:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.4|       |
|                Type|             MANAGED|       |
|            Provider|             parquet|       |
|            Location|file:/C:/Users/aa...|       |
+--------------------+--------------------+-------+


,employee_key,employee_id
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


### loading customers json file into mongodb collection

In [155]:
client = get_mongo_client(**mongodb_args)

json_files = {
    "customers": "dim_customers.json"
}

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files)
print("MongoDB collection loaded successfully.")

MongoDB collection loaded successfully.


### now fetch customers data from mongodb

In [156]:
mongodb_args["collection"] = "customers"

df_dim_customers = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_customers.toPandas().head(2)

,AccountNumber,AddressLine1,AddressType,City,CountryRegionCode,Country_Region,CustomerID,CustomerType,IsOnlyStateProvinceFlag,PostalCode,Sales Territory,Sales Territory Group,StateProvinceCode,State_Province
0,AW00000001,2251 Elliot Avenue,Main Office,Seattle,US,United States,1,S,0,98104,Northwest,North America,WA,Washington
1,AW00000002,7943 Walnut Ave,Shipping,Renton,US,United States,2,S,0,98055,Northwest,North America,WA,Washington


### transformations

In [157]:
df_dim_customers = df_dim_customers.withColumnRenamed("CustomerID", "customer_id")

#surrogate primary key
df_dim_customers.createOrReplaceTempView("customers")
sql_customers = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key
    FROM customers;
"""
df_dim_customers = spark.sql(sql_customers)

ordered_columns = ['customer_key', 'customer_id', 'first_name', 'last_name',
                   'email', 'phone', 'address', 'city', 'state', 'country']

existing_columns = [c for c in ordered_columns if c in df_dim_customers.columns]
df_dim_customers = df_dim_customers[existing_columns]
df_dim_customers.toPandas().head(2)

,customer_key,customer_id
0,1,1
1,2,2


In [158]:
df_dim_customers

DataFrame[customer_key: int, customer_id: int]

### save as dim_customers in the lakehouse

In [159]:
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

### describe+preview table

In [160]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 5").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        customer_key|                 int|   NULL|
|         customer_id|                 int|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|       dim_customers|       |
|        Created Time|Fri May 08 23:43:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.4|       |
|                Type|             MANAGED|       |
|            Provider|             parquet|       |
|            Location|file:/C:/Users/aa...|       |
+--------------------+--------------------+-------+


,customer_key,customer_id
0,1,1
1,2,2
2,3,2
3,4,3
4,5,4


### fetch data from mysql dimdate

In [161]:
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)
df_dim_date.toPandas().head(2)

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


### add to table

In [162]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

### preview table

In [163]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 5").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            date_key|      int|   NULL|
|           full_date|     date|   NULL|
|           date_name| char(11)|   NULL|
|        date_name_us| char(11)|   NULL|
|        date_name_eu| char(11)|   NULL|
|         day_of_week|  tinyint|   NULL|
|    day_name_of_week| char(10)|   NULL|
|        day_of_month|  tinyint|   NULL|
|         day_of_year|      int|   NULL|
|     weekday_weekend| char(10)|   NULL|
|        week_of_year|  tinyint|   NULL|
|          month_name| char(10)|   NULL|
|       month_of_year|  tinyint|   NULL|
|is_last_day_of_month|  char(1)|   NULL|
|    calendar_quarter|  tinyint|   NULL|
|       calendar_year|      int|   NULL|
| calendar_year_month| char(10)|   NULL|
|   calendar_year_qtr| char(10)|   NULL|
|fiscal_month_of_year|  tinyint|   NULL|
|      fiscal_quarter|  tinyint|   NULL|
+--------------------+---------+-------+


,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
2,20000103,2000-01-03,2000/01/03,01/03/2000,03/01/2000,2,Monday,3,3,Weekday,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
3,20000104,2000-01-04,2000/01/04,01/04/2000,04/01/2000,3,Tuesday,4,4,Weekday,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
4,20000105,2000-01-05,2000/01/05,01/05/2000,05/01/2000,4,Wednesday,5,5,Weekday,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


### fetch data from the Products table in MySQL

In [164]:
sql_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY ProductID) AS product_key
    FROM {mysql_args['db_name']}.product
"""
df_dim_products = get_mysql_dataframe(spark, sql_products, **mysql_args)
df_dim_products.toPandas().head(2)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate,product_key
0,1,Adjustable Race,AR-5381,False,False,NaN,1000,750,0.0,0.0,...,NaN,NaN,NaN,NaN,1998-06-01,NaT,NaT,"[183, 21, 66, 105, 247, 8, 13, 76, 172, 177, 2...",2004-03-11 10:01:36,1
1,2,Bearing Ball,BA-8327,False,False,NaN,1000,750,0.0,0.0,...,NaN,NaN,NaN,NaN,1998-06-01,NaT,NaT,"[32, 60, 174, 88, 58, 79, 73, 71, 167, 212, 21...",2004-03-11 10:01:36,2


### transformations

In [165]:
# rename productid
df_dim_products = df_dim_products.withColumnRenamed("ProductID", "product_id")

# drop some large text columns
for drop_col in ['description', 'LargePhoto', 'ThumbNailPhoto', 'ThumbnailPhotoFileName', 'rowguid', 'ModifiedDate']:
    if drop_col in df_dim_products.columns:
        df_dim_products = df_dim_products.drop(drop_col)

df_dim_products.toPandas().head(2)

,product_id,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,product_key
0,1,Adjustable Race,AR-5381,False,False,NaN,1000,750,0.0,0.0,...,0,NaN,NaN,NaN,NaN,NaN,1998-06-01,NaT,NaT,1
1,2,Bearing Ball,BA-8327,False,False,NaN,1000,750,0.0,0.0,...,0,NaN,NaN,NaN,NaN,NaN,1998-06-01,NaT,NaT,2


### save as dim_products

In [166]:
df_dim_products.write.saveAsTable(f"{dest_database}.dim_products", mode="overwrite")

### preview table

In [167]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_products;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_products LIMIT 5").toPandas()

+--------------------+------------+-------+
|            col_name|   data_type|comment|
+--------------------+------------+-------+
|          product_id|         int|   NULL|
|                Name| varchar(50)|   NULL|
|       ProductNumber| varchar(25)|   NULL|
|            MakeFlag|     boolean|   NULL|
|   FinishedGoodsFlag|     boolean|   NULL|
|               Color| varchar(15)|   NULL|
|    SafetyStockLevel|         int|   NULL|
|        ReorderPoint|         int|   NULL|
|        StandardCost|      double|   NULL|
|           ListPrice|      double|   NULL|
|                Size|  varchar(5)|   NULL|
| SizeUnitMeasureCode|  varchar(3)|   NULL|
|WeightUnitMeasure...|  varchar(3)|   NULL|
|              Weight|decimal(8,2)|   NULL|
|   DaysToManufacture|         int|   NULL|
|         ProductLine|  varchar(2)|   NULL|
|               Class|  varchar(2)|   NULL|
|               Style|  varchar(2)|   NULL|
|ProductSubcategoryID|         int|   NULL|
|      ProductModelID|         i

,product_id,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,product_key
0,1,Adjustable Race,AR-5381,False,False,None,1000,750,0.0,0.0,...,0,None,None,None,NaN,NaN,1998-06-01,NaT,NaT,1
1,2,Bearing Ball,BA-8327,False,False,None,1000,750,0.0,0.0,...,0,None,None,None,NaN,NaN,1998-06-01,NaT,NaT,2
2,3,BB Ball Bearing,BE-2349,True,False,None,800,600,0.0,0.0,...,1,None,None,None,NaN,NaN,1998-06-01,NaT,NaT,3
3,4,Headset Ball Bearings,BE-2908,False,False,None,800,600,0.0,0.0,...,0,None,None,None,NaN,NaN,1998-06-01,NaT,NaT,4
4,316,Blade,BL-2036,True,False,None,800,600,0.0,0.0,...,1,None,None,None,NaN,NaN,1998-06-01,NaT,NaT,5


### verify all dimension tables

In [168]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,dim_employees,False
3,adventureworks_dlh,dim_products,False
4,,customers,True
5,,employees,True



### info of the source streaming files

In [169]:
get_file_info(sales_orders_stream_dir)

,name,size,modification_time
0,sales_orders_01.json,15208448,2026-05-08 09:42:00.351876497
1,sales_orders_02.json,1146880,2026-05-08 09:43:48.872922659
2,sales_orders_03.json,41802302,2026-05-08 09:56:31.872737408


### bronze layer: read raw json files into a stream

In [170]:
df_sales_orders_bronze = (
    spark.readStream
    .option("schemaLocation", sales_orders_output_bronze)
    .option("maxFilesPerTrigger", 1)
    .option("multiLine", "true")
    .json(sales_orders_stream_dir)
)

df_sales_orders_bronze.isStreaming

True

### bronze layer: write streaming data to a parquet file

In [171]:
sales_orders_checkpoint_bronze = os.path.join(sales_orders_output_bronze, '_checkpoint')

sales_orders_bronze_query = (
    df_sales_orders_bronze
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    .writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("sales_orders_bronze")
    .trigger(availableNow=True)
    .option("checkpointLocation", sales_orders_checkpoint_bronze)
    .option("compression", "snappy")
    .start(sales_orders_output_bronze)
)

### test the bronze query

In [172]:
print(f"Query ID:     {sales_orders_bronze_query.id}")
print(f"Query Name:   {sales_orders_bronze_query.name}")
print(f"Query Status: {sales_orders_bronze_query.status}")

Query ID:     ea6f1ee5-19f5-4984-a1cc-11616d5d4796
Query Name:   sales_orders_bronze
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [173]:
sales_orders_bronze_query.awaitTermination()

### silver layer: prepare role playing date dim views

In [174]:
df_dim_order_date = df_dim_date.select(
    col("date_key").alias("order_date_key"),
    col("full_date").alias("order_full_date")
)

df_dim_ship_date = df_dim_date.select(
    col("date_key").alias("ship_date_key"),
    col("full_date").alias("ship_full_date")
)

df_dim_due_date = df_dim_date.select(
    col("date_key").alias("due_date_key"),
    col("full_date").alias("due_full_date")
)

##### silver layer: create silver query+join streaming data with dim tables

In [175]:
df_sales_orders_silver = (
    spark.readStream.format("parquet").load(sales_orders_output_bronze)
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("SalesPersonID", "employee_id")
    .withColumnRenamed("ProductID", "product_id")
    .withColumnRenamed("SalesOrderID", "sales_order_id")
    .withColumnRenamed("OrderDate", "order_date")
    .withColumnRenamed("ShipDate", "ship_date")
    .withColumnRenamed("DueDate", "due_date")
    .withColumnRenamed("OrderQty", "order_qty")
    .withColumnRenamed("UnitPrice", "unit_price")
    .withColumnRenamed("LineTotal", "line_total")
    .withColumnRenamed("OnlineOrderFlag", "online_order_flag")
    .withColumnRenamed("Status", "status")
    .join(df_dim_customers, "customer_id", "inner")
    .join(df_dim_employees, "employee_id", "left_outer")
    .join(df_dim_products, "product_id", "inner")
    .join(
        df_dim_order_date,
        df_dim_order_date.order_full_date.cast(DateType()) == col("order_date").cast(DateType()),
        "inner"
    )
    .join(
        df_dim_ship_date,
        df_dim_ship_date.ship_full_date.cast(DateType()) == col("ship_date").cast(DateType()),
        "left_outer"
    )
    .join(
        df_dim_due_date,
        df_dim_due_date.due_full_date.cast(DateType()) == col("due_date").cast(DateType()),
        "left_outer"
    )
    .select(
        col("sales_order_id").cast(LongType()),
        df_dim_customers.customer_key.cast(LongType()),
        df_dim_employees.employee_key.cast(LongType()),
        df_dim_products.product_key.cast(LongType()),
        df_dim_order_date.order_date_key.cast(LongType()),
        df_dim_ship_date.ship_date_key.cast(LongType()),
        df_dim_due_date.due_date_key.cast(LongType()),
        col("order_qty").cast(IntegerType()),
        col("unit_price").cast(DoubleType()),
        col("line_total").cast(DoubleType()),
        col("status"),
        col("online_order_flag")
    )
)

df_sales_orders_silver.isStreaming

True

In [176]:
print("Customers:", df_dim_customers.columns)
print("Employees:", df_dim_employees.columns)
print("Products:", df_dim_products.columns)

Customers: ['customer_key', 'customer_id']
Employees: ['employee_key', 'employee_id']
Products: ['product_id', 'Name', 'ProductNumber', 'MakeFlag', 'FinishedGoodsFlag', 'Color', 'SafetyStockLevel', 'ReorderPoint', 'StandardCost', 'ListPrice', 'Size', 'SizeUnitMeasureCode', 'WeightUnitMeasureCode', 'Weight', 'DaysToManufacture', 'ProductLine', 'Class', 'Style', 'ProductSubcategoryID', 'ProductModelID', 'SellStartDate', 'SellEndDate', 'DiscontinuedDate', 'product_key']


In [177]:
df_sales_orders_silver.printSchema()

root
 |-- sales_order_id: long (nullable = true)
 |-- customer_key: long (nullable = false)
 |-- employee_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- order_date_key: long (nullable = true)
 |-- ship_date_key: long (nullable = true)
 |-- due_date_key: long (nullable = true)
 |-- order_qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- status: long (nullable = true)
 |-- online_order_flag: string (nullable = true)


### add transformed data to table

In [178]:
sales_orders_checkpoint_silver = os.path.join(sales_orders_output_silver, '_checkpoint')

sales_orders_silver_query = (
    df_sales_orders_silver.writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("sales_orders_silver")
    .trigger(availableNow=True)
    .option("checkpointLocation", sales_orders_checkpoint_silver)
    .option("compression", "snappy")
    .start(sales_orders_output_silver)
)

### test silver query

In [179]:
print(f"Query ID:     {sales_orders_silver_query.id}")
print(f"Query Name:   {sales_orders_silver_query.name}")
print(f"Query Status: {sales_orders_silver_query.status}")

Query ID:     cd06356c-7ae2-4f83-9465-0cad63c3818d
Query Name:   sales_orders_silver
Query Status: {'message': 'Writing offsets to log', 'isDataAvailable': False, 'isTriggerActive': True}


In [180]:
sales_orders_silver_query.awaitTermination()

### gold layer: business relevant query

In [181]:
df_sales_by_category_gold = (
    spark.readStream.format("parquet").load(sales_orders_output_silver)
    .join(df_dim_products, "product_key")
    .join(
        df_dim_date,
        df_dim_date.date_key.cast(LongType()) == col("order_date_key").cast(LongType())
    )
    .groupBy("month_of_year", "month_name", "ProductLine")
    .agg(
        count("product_key").alias("units_sold"),
        round(sum("line_total"), 2).alias("total_revenue")
    )
    .orderBy(asc("month_of_year"), desc("total_revenue"))
)

In [182]:
df_dim_products

DataFrame[product_id: int, Name: string, ProductNumber: string, MakeFlag: boolean, FinishedGoodsFlag: boolean, Color: string, SafetyStockLevel: int, ReorderPoint: int, StandardCost: double, ListPrice: double, Size: string, SizeUnitMeasureCode: string, WeightUnitMeasureCode: string, Weight: decimal(8,2), DaysToManufacture: int, ProductLine: string, Class: string, Style: string, ProductSubcategoryID: int, ProductModelID: int, SellStartDate: timestamp, SellEndDate: timestamp, DiscontinuedDate: timestamp, product_key: decimal(20,0)]

### complete gold query

In [183]:
sales_gold_query = (
    df_sales_by_category_gold.writeStream
    .format("memory")
    .outputMode("complete")
    .queryName("fact_sales_by_category")
    .start()
)

wait_until_stream_is_ready(sales_gold_query, 1)

The stream has processed 1 batch(es)


### querying from gold data

In [184]:
df_fact_sales_by_category = spark.sql("SELECT * FROM fact_sales_by_category")
df_fact_sales_by_category.printSchema()

root
 |-- month_of_year: byte (nullable = true)
 |-- month_name: string (nullable = true)
 |-- ProductLine: string (nullable = true)
 |-- units_sold: long (nullable = false)
 |-- total_revenue: double (nullable = true)


### final select statement, organize sales by month

In [185]:
df_fact_sales_by_category_final = (
    df_fact_sales_by_category
    .select(
        col("month_name").alias("Month"),
        col("ProductLine").alias("Product Line"),
        col("units_sold").alias("Units Sold"),
        col("total_revenue").alias("Total Revenue ($)")
    )
    .orderBy(asc("month_of_year"), desc("Total Revenue ($)"))
)

### save to the data lakehouse and display

In [186]:
df_fact_sales_by_category_final.write.saveAsTable(
    f"{dest_database}.fact_sales_by_product_category", mode="overwrite"
)
spark.sql(f"SELECT * FROM {dest_database}.fact_sales_by_product_category").toPandas()

,Month,Product Line,Units Sold,Total Revenue ($)
0,January,R,7,79.04
1,February,R,576,918463.18
2,February,M,649,892650.20
3,June,M,903,1106386.97
4,June,R,559,863098.07
5,June,S,715,99475.10
6,January,T,1,2384.07
7,January,S,11,229.76
8,January,M,6,136.96
9,April,R,475,790102.17


In [187]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,dim_employees,False
3,adventureworks_dlh,dim_products,False
4,adventureworks_dlh,fact_sales_by_product_category,False
5,,customers,True
6,,employees,True
7,,fact_sales_by_category,True


### 7.0. Stop the Spark Session

In [188]:
spark.stop()